# Romberg rule

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differentiation_and_integration/integration/romberg_rule.ipynb)

In [ ]:
import numpy as np
import scipy as sp
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Richardson Extrapolation

An alternative to the Newton-Cotes formulae uses successive sampling step sizes to estimate and cancel error. We begin with Richardson extrapolation and extend this to Romberg integration.

Richardson extrapolation is an algorithm that uses successive applications of the trapezoid rule with differing step sizes to achieve superior accuracy without computing higher derivatives.

The exact integral can always be expressed as:

$$I = I(h) + E(h)$$

where $I(h)$ is the approximation with step $h$ and $E(h)$ is the associated truncation error. Writing this for step $h_1$ and $h_2 = h_1 / 2$:

$$
\begin{aligned}
I &= I(h_1) + c h_1^2 + O(h_1^4) \\
I &= I(h_2) + c h_2^2 + O(h_2^4) = I(h_2) + c \frac{h_1^2}{4} + O(h_1^4)
\end{aligned}
$$

Multiplying the second equation by 4 and subtracting the first eliminates the leading $h_1^2$ error term:

$$I = \frac{4 I(h_2) - I(h_1)}{3} + O(h_1^4)$$

This is an interesting result! Effectively what we have done is use a second estimate to cancel the next power in our expansion, leading to a higher-order result without evaluating new derivatives.

## The Romberg Integration Algorithm

In fact, we can repeat this procedure arbitrarily! Above we combined two $O(h^2)$ estimates to make an $O(h^4)$ estimate. We can take this result, combine it with another, and make an $O(h^6)$ estimate, and so on.

This recursive extrapolation generates the Romberg tableau:

$$R_{j, k} = \frac{4^k R_{j, k-1} - R_{j-1, k-1}}{4^k - 1}$$

where $j$ is the row (grid refinement step $h_j = (b-a)/2^j$) and $k$ is the column (extrapolation order).

## Example: Romberg Integration vs. Simpson''s Rule

Approximate $\int_0^\pi \sin(x) \,\mathrm{d}x = 2$ using the Romberg rule and compare with Simpson''s 1/3 rule:

In [ ]:
def f(x):
    return np.sin(x)

for n in [4, 8, 16, 32, 64, 128]:
    x = np.linspace(0, np.pi, n + 1)
    f_x = f(x)
    romberg_val = sp.integrate.romb(f_x, dx=np.pi / n)
    simpson_val = sp.integrate.simpson(f_x, x=x)
    print(f"Intervals n={n:3d} | Romberg error: {abs(romberg_val - 2):.2e} | Simpson error: {abs(simpson_val - 2):.2e}")

## Convergence to Machine Precision

Note the error became zero (or reached double-precision floating point limits $\approx 10^{-16}$)! What does that mean?

By systematically canceling higher-order polynomial truncation terms, Romberg integration converges exponentially for smooth functions with very few function evaluations.

In [ ]:
fig = exe.plot_romberg_convergence(np.sin, 0, np.pi)
fig.show()